# Moni Pipeline Demo

기존 `moni_pipeline.ipynb`를 DB 테이블 기반 AI 엔진으로 전환한 후의 데모.

**핵심 변경**:
- 데이터 생성 로직을 엔진에서 분리 (시드 CSV 또는 BE가 주입)
- `final_category` 기준 일 단위 시계열 생성
- '30일 합계'가 아닌 '이번 달 월말 예상 지출' 계산
- 카테고리명 변수화 (카페 하드코딩 제거)
- `User_Category_Settings.budget_limit` 기반 압박도
- 압박도 최고 카테고리 1개를 오늘의 챌린지로 선택
- `Daily_Challenges` 저장용 JSON 반환

In [ ]:
# Colab에서 실행 시
# !pip install prophet pandas numpy --quiet

import sys
from pathlib import Path

# 노트북에서 moni_engine 패키지를 찾을 수 있도록 경로 추가
sys.path.insert(0, str(Path.cwd().parent))

import json
from datetime import date
import pandas as pd

## 1. 시드 데이터 로드

실제로는 BE가 DB에서 조회해 넘겨주는 데이터. MVP 단계에서는 시드 CSV로 대체.

In [ ]:
SEED_DIR = Path.cwd().parent / "seed_data"

transactions_df = pd.read_csv(SEED_DIR / "seed_transactions.csv")
users_df = pd.read_csv(SEED_DIR / "seed_users.csv")
category_settings_df = pd.read_csv(SEED_DIR / "seed_category_settings.csv")

print(f"transactions: {len(transactions_df)} rows")
print(f"users:        {len(users_df)} rows")
print(f"settings:     {len(category_settings_df)} rows")
print()
print("--- 카테고리별 거래 건수 ---")
print(transactions_df["final_category"].value_counts())

## 2. AI 엔진 호출

**이 한 줄이 백엔드가 FastAPI에서 호출할 진입점**.

In [ ]:
from moni_engine.engine import get_today_challenge

user_profile = users_df.iloc[0].to_dict()
target_date = date(2026, 5, 14)

result = get_today_challenge(
    transactions_df=transactions_df,
    user_profile=user_profile,
    category_settings_df=category_settings_df,
    target_date=target_date,
)

print(json.dumps(result, ensure_ascii=False, indent=2, default=str))

## 3. Daily_Challenges 저장 형태 미리보기

BE 입장에서는 결과 dict의 최상위 필드를 `Daily_Challenges` 컬럼으로, `ai_metadata`를 JSON 컬럼으로 INSERT하면 됨.

In [ ]:
from IPython.display import HTML, display

if result is None:
    print("오늘은 챌린지가 없습니다.")
else:
    md = result["ai_metadata"]
    diff_color = {"Easy": "#2e7d32", "Medium": "#ef6c00",
                  "Medium-Hard": "#e64a19", "Hard": "#d32f2f"}[result["difficulty"]]

    html = f"""
    <div style='font-family:-apple-system,"Segoe UI",Arial,sans-serif;
                max-width:520px; border:1px solid #e6e6e6; border-radius:18px;
                padding:20px; box-shadow:0 2px 10px rgba(0,0,0,0.04); background:white'>
        <div style='display:inline-block; background:{diff_color}; color:white;
                    padding:5px 12px; border-radius:999px; font-size:12px;
                    font-weight:700; margin-bottom:12px;'>
            {result['difficulty']} · {result['challenge_type']}
        </div>
        <div style='font-size:22px; font-weight:800; margin-bottom:10px; line-height:1.4;'>
            {result['challenge_text']}
        </div>
        <div style='font-size:13px; color:#666; margin-bottom:14px;'>
            {md['reason']}
        </div>
        <div style='display:grid; grid-template-columns:1fr 1fr; gap:8px; font-size:13px;'>
            <div><span style='color:#666'>카테고리</span> <b>{result['category_name']}</b></div>
            <div><span style='color:#666'>XP 보상</span> <b>{result['xp_reward']}</b></div>
            <div><span style='color:#666'>월 예산</span> <b>{int(md['budget_limit']):,}원</b></div>
            <div><span style='color:#666'>예상 월 지출</span> <b>{int(md['predicted_monthly_spend']):,}원</b></div>
            <div><span style='color:#666'>현재까지 실지출</span> <b>{int(md['month_to_date_actual']):,}원</b></div>
            <div><span style='color:#666'>예측 잔여</span> <b>{int(md['predicted_remaining_spend']):,}원</b></div>
            <div><span style='color:#666'>압박도</span> <b>{md['budget_pressure']:.2f}배</b></div>
            <div><span style='color:#666'>모델</span> <b>{md['model_used']}</b></div>
        </div>
    </div>
    """
    display(HTML(html))

## 4. 후보 카테고리 비교

`ai_metadata.evaluated_categories`에 모든 후보의 압박도가 들어있어서 발표/디버깅 시 참고 가능.

In [ ]:
if result is not None:
    eval_df = pd.DataFrame(result["ai_metadata"]["evaluated_categories"])
    display(eval_df)

## 5. 다른 시나리오 (다른 날짜)

`target_date`만 바꾸면 동일 데이터로 다른 날의 챌린지 생성 가능 (백테스팅용).

In [ ]:
for d in [date(2026, 5, 2), date(2026, 5, 14), date(2026, 5, 28)]:
    r = get_today_challenge(transactions_df, user_profile, category_settings_df, d)
    if r is None:
        print(f"{d}: NO CHALLENGE")
        continue
    md = r["ai_metadata"]
    print(f"{d} | {r['category_name']:4s} | pressure={md['budget_pressure']:.2f} | "
          f"{r['difficulty']:11s} | {r['challenge_text']}")